# APTOS — Test eval fix (loads existing checkpoint)

**Why this notebook exists:** the training notebook (`aptos-baseline` or similar slug) ran for 88 minutes and trained successfully (best val QWK = 0.9010 at epoch 15) but crashed in its final test-eval cell because PyTorch 2.6's new default `weights_only=True` doesn't accept checkpoints containing numpy scalars. The checkpoint itself is fine. We just re-load it here with `weights_only=False` and finish the test eval.

**Inputs to attach (right sidebar → + Add Input):**
1. `mariaherrerot/aptos-2019-dataset` — same APTOS dataset
2. **Notebook Output:** your APTOS training notebook (whatever you named it on Kaggle, likely `samarthmishra208/aptos-baseline` or similar)

**Settings:** GPU T4 (test eval still needs ~3 min GPU inference, can use CPU but slower). Internet off.

**Runtime:** ~5-10 minutes total.

## Cell 1 — Imports + locate inputs

In [ ]:
import os, sys, json, time, glob, pathlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import (roc_auc_score, accuracy_score, confusion_matrix,
                              cohen_kappa_score, classification_report)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "GPU:", torch.cuda.is_available(),
      "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# Find the training notebook output (the directory containing best.pt)
ckpt_candidates = glob.glob("/kaggle/input/**/best.pt", recursive=True)
print("\nFound best.pt at:")
for c in ckpt_candidates:
    print(" ", c)
assert ckpt_candidates, "best.pt not found — attach your aptos training notebook output"
CKPT = ckpt_candidates[0]

# APTOS data paths
ROOT = "/kaggle/input/datasets/mariaherrerot/aptos2019"
TEST_CSV = os.path.join(ROOT, "test.csv")
TEST_IMG = os.path.join(ROOT, "test_images", "test_images")
for p in [TEST_CSV, TEST_IMG]:
    assert os.path.exists(p), f"Missing: {p}"
print(f"\nCKPT: {CKPT}")
print(f"TEST_CSV: {TEST_CSV}")
print(f"TEST_IMG: {TEST_IMG}")

WORK = pathlib.Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)

## Cell 2 — Dataset + transforms + model definitions

In [ ]:
CLASSES = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative"]
N_CLASSES = len(CLASSES)
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class APTOSTestDataset(Dataset):
    def __init__(self, csv, img_dir, transform=None):
        self.df = pd.read_csv(csv).reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.labels = self.df["diagnosis"].astype(int).to_numpy()
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, f"{row['id_code']}.png")).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, int(self.labels[idx]), row["id_code"]

def build_resnet50(num_classes=N_CLASSES, dropout=0.3):
    m = models.resnet50(weights=None)
    in_feat = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(in_feat, num_classes))
    return m

test_ds = APTOSTestDataset(TEST_CSV, TEST_IMG, transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f"Test dataset: {len(test_ds):,} images")

## Cell 3 — Load checkpoint with `weights_only=False`

This is the one-character fix. `weights_only=False` accepts numpy scalars in the checkpoint metadata. Safe to use here because we generated the checkpoint ourselves.

In [ ]:
state = torch.load(CKPT, map_location=DEVICE, weights_only=False)
model = build_resnet50().to(DEVICE)
model.load_state_dict(state["model_state"])
model.eval()
print(f"Loaded checkpoint epoch {state['epoch']+1}")
print(f"  val_qwk: {state.get('val_qwk')}")
print(f"  val_auroc: {state.get('val_auroc')}")

## Cell 4 — Run test eval + bootstrap CIs + save

In [ ]:
@torch.no_grad()
def run_inference(model, loader, device):
    all_logits, all_labels = [], []
    for x, y, _ in loader:
        x = x.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(x)
        all_logits.append(logits.float().cpu())
        all_labels.append(y)
    return torch.cat(all_logits), torch.cat(all_labels)

print("Running test inference...")
t0 = time.time()
test_logits, test_labels = run_inference(model, test_loader, DEVICE)
print(f"  {len(test_labels):,} predictions in {time.time()-t0:.1f}s")

test_probs = F.softmax(test_logits, dim=1).numpy()
y_true = test_labels.numpy()
y_pred = test_probs.argmax(axis=1)

# Headline metrics
test_acc = accuracy_score(y_true, y_pred)
test_qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
aurocs = []
per_class_auroc = {}
for i, c in enumerate(CLASSES):
    y_bin = (y_true == i).astype(int)
    if y_bin.sum() > 0:
        a = roc_auc_score(y_bin, test_probs[:, i])
        aurocs.append(a)
        per_class_auroc[c] = float(a)
    else:
        per_class_auroc[c] = None
test_auroc_macro = float(np.mean(aurocs))
cm = confusion_matrix(y_true, y_pred, labels=list(range(N_CLASSES)))

# Bootstrap CIs
def bootstrap_ci(probs, y_true, n_boot=1000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    accs, qwks, macro_aurocs = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        pb, yb = probs[idx], y_true[idx]
        accs.append(accuracy_score(yb, pb.argmax(axis=1)))
        qwks.append(cohen_kappa_score(yb, pb.argmax(axis=1), weights='quadratic'))
        aurocs = []
        for c in range(N_CLASSES):
            ybin = (yb == c).astype(int)
            if ybin.sum() == 0 or ybin.sum() == len(ybin): continue
            aurocs.append(roc_auc_score(ybin, pb[:, c]))
        if aurocs: macro_aurocs.append(np.mean(aurocs))
    def ci(v):
        v = np.asarray(v)
        return [float(np.quantile(v, alpha/2)), float(np.quantile(v, 1-alpha/2))]
    return ci(accs), ci(qwks), ci(macro_aurocs)

acc_ci, qwk_ci, auroc_ci = bootstrap_ci(test_probs, y_true)

results = {
    "test_n": int(len(y_true)),
    "test_accuracy": float(test_acc),
    "test_accuracy_ci95": acc_ci,
    "test_qwk": float(test_qwk),
    "test_qwk_ci95": qwk_ci,
    "test_auroc_macro": float(test_auroc_macro),
    "test_auroc_macro_ci95": auroc_ci,
    "test_auroc_per_class": per_class_auroc,
    "confusion_matrix": cm.tolist(),
    "classes": CLASSES,
    "best_epoch": int(state["epoch"] + 1),
    "best_val_qwk": float(state.get("val_qwk", 0)),
}
with open(RESULTS / "test_metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n=== Test results ===")
print(json.dumps(results, indent=2))

print("\nConfusion matrix (rows=true, cols=pred):")
print("                 " + "  ".join(f"{c:>14s}" for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f"  {c:>13s}  " + "  ".join(f"{cm[i,j]:>14d}" for j in range(N_CLASSES)))

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

import shutil
zip_path = WORK / "aptos_eval_results.zip"
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(RESULTS))
print(f"\nWrote {zip_path}")